In [4]:
# ===============================
# import libraries
# ===============================
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# ===============================
# import data
# ===============================

# set data directory
DATA_DIR = Path("data/sets")

# combine every saved set
pokemon_all = pd.concat(
    [pd.read_csv(path, dtype={"number": str, "product_id": str, "variant": str}, parse_dates=["date"]) for path in DATA_DIR.glob("*.csv")],
    ignore_index=True
)

# additional data on each set
sets_df = pd.read_csv("data/sets_info.csv", parse_dates=["release_date"])
pokemon_all = pokemon_all.merge(sets_df, on="set_slug", how="left")

,grade,date,price,set_slug,product_id,name,variant,number,set_name,language,release_date,promo
0,Grade 9.5,2022-06-01 06:00:00,0.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,NaN,42,Pokemon 1999 Topps Movie,English,1999-06-01,False
1,Grade 9.5,2022-07-01 06:00:00,0.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,NaN,42,Pokemon 1999 Topps Movie,English,1999-06-01,False
2,Grade 9.5,2022-08-01 06:00:00,0.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,NaN,42,Pokemon 1999 Topps Movie,English,1999-06-01,False
3,Grade 9.5,2022-09-01 06:00:00,0.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,NaN,42,Pokemon 1999 Topps Movie,English,1999-06-01,False
4,Grade 9.5,2022-10-01 06:00:00,0.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,NaN,42,Pokemon 1999 Topps Movie,English,1999-06-01,False
...,...,...,...,...,...,...,...,...,...,...,...,...
13124125,Ungraded,2026-06-01 06:00:00,0.52,pokemon-xy,806728,bunnelby,Reverse Holo,111,Pokemon XY,English,2014-02-05,False
13124126,Ungraded,2026-07-01 06:00:00,0.70,pokemon-xy,806728,bunnelby,Reverse Holo,111,Pokemon XY,English,2014-02-05,False
13124127,Ungraded,2026-08-01 06:00:00,0.66,pokemon-xy,806728,bunnelby,Reverse Holo,111,Pokemon XY,English,2014-02-05,False
13124128,Ungraded,2026-09-01 06:00:00,0.64,pokemon-xy,806728,bunnelby,Reverse Holo,111,Pokemon XY,English,2014-02-05,False


In [ ]:
# ===============================
# clean data
# ===============================
pokemon_all["variant"] = pokemon_all["variant"].fillna("Standard")
pokemon_all["price"] = pokemon_all["price"].replace(0, np.nan)
pokemon_all["date"] = pokemon_all["date"].dt.to_period("M").dt.to_timestamp()
pokemon_all["decade"] = (pokemon_all["release_date"].dt.year // 10) * 10


for col in ["grade", "set_slug", "set_name", "language", "name", "variant"]:
    pokemon_all[col] = pokemon_all[col].astype("category")



In [ ]:
# ===============================
# Cards from latest date only
# ===============================
latest_cards  = (
    pokemon_all
    .filter("date" == "2026-10-01")
    .sort_values("date", ascending=False)
    .drop_duplicates(subset="product_id", keep="first")
)

latest_cards

duplicates = latest_cards[latest_cards.duplicated(subset=['product_id'], keep=False)]
print(duplicates.sort_values('product_id'))

# 1. Count how many different values exist (ignores NaNs by default)
count = latest_cards['date'].nunique()
print(f"Number of unique values: {count}")

# 2. See what those unique values are
unique_values = latest_cards['date'].unique()
print(unique_values)

In [ ]:
pokemon_all.info()

# pokemon_all["variant"].astype(str).drop_duplicates().to_frame()


In [ ]:
variant_counts = (
    pokemon_all.groupby("variant")["product_id"]
    .nunique()
    .sort_index(ascending=True)
)

variant_counts

In [ ]:
variant_info = pokemon_all[pokemon_all["variant"] == "Toys R Us Promo"]

variant_info

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, LogLocator

import chartgazing as cg

VALUE, GROUP, NAME = "price", "decade", "set_name"

unit_prefix = "$"
marker_size = 5.5


def swarm(x, diameter):
    """y offsets (same units as x) so dots of the given diameter don't overlap. Greedy, centre-out."""
    order = np.argsort(x)
    ys = np.zeros(len(x))
    placed_x, placed_y = [], []
    for i in order:
        xi = x[i]
        near = [(xj, yj) for xj, yj in zip(placed_x, placed_y) if abs(xj - xi) < diameter]
        candidates = [0.0]
        for xj, yj in near:
            dy = np.sqrt(diameter ** 2 - (xi - xj) ** 2)
            candidates += [yj + dy, yj - dy]
        for c in sorted(candidates, key=abs):
            if all((xi - xj) ** 2 + (c - yj) ** 2 >= diameter ** 2 - 1e-9 for xj, yj in near):
                ys[i] = c
                break
        placed_x.append(xi)
        placed_y.append(ys[i])
    return ys


def set_price_comparison(df, grade="Ungraded", variant="Standard", stat="median"):
    latest = df[
        (df["variant"] == variant) 
        & (df["grade"] == grade)
        & (df["date"] == df["date"].max())
        ]

    set_prices = (
        latest.groupby("set_name", observed=True)
        .agg(price=("price", stat), decade=("decade", "first"))
        .dropna(subset=["decade", "price"])
        .reset_index()
    )
    set_prices["decade"] = set_prices["decade"].astype(int).astype(str) + "s"

    return set_prices.sort_values('price', ascending=False)

def decade_beeswarm(df, title, subtitle, path, highlight=(), note="", source="", pad=1.25):

    order = sorted(df[GROUP].unique(), reverse=True)   # 1990s on top

    fig, ax = cg.chart(title, subtitle, note=note, source=source, grid="x")

    ax.set_xscale("log")
    ax.set_xlim(df[VALUE].min() / pad, df[VALUE].max() * pad)
    ax.xaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5), numticks=100))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{unit_prefix}{x:,g}"))
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.set_ylim(-0.6, len(order) - 0.4)
    ax.set_yticks(range(len(order)), order)

    # the swarm is laid out in points, so fit the layout first to know the final axes size
    # on a log axis, distance is measured in log10 units
    cg.fit(fig)
    bbox = ax.get_window_extent()
    pt = 72 / fig.dpi
    x_per_pt = np.diff(np.log10(ax.get_xlim()))[0] / (bbox.width * pt)
    y_per_pt = np.diff(ax.get_ylim())[0] / (bbox.height * pt)

    for yi, group_name in enumerate(order):
        group = df[df[GROUP] == group_name]
        offsets = swarm(np.log10(group[VALUE].to_numpy()) / x_per_pt, marker_size * 1.05) * y_per_pt
        is_hl = group[NAME].isin(highlight).to_numpy()

        ax.scatter(group[VALUE][~is_hl], yi + offsets[~is_hl], s=marker_size ** 2,
                color=cg.SECONDARY, alpha=0.75, linewidth=0, clip_on=False)
        ax.scatter(group[VALUE][is_hl], yi + offsets[is_hl], s=(marker_size * 1.3) ** 2,
                color=cg.PRIMARY, edgecolor=cg.BACKGROUND, linewidth=0.6, zorder=3, clip_on=False)

        for name, xv, yo in zip(group[NAME][is_hl], group[VALUE][is_hl], offsets[is_hl]):
            ax.annotate(name, xy=(xv, yi + yo), xytext=(0, 6), textcoords="offset points",
                        ha="center", va="bottom", fontsize=7.5, fontweight="bold", color=cg.TEXT)

    cg.save(fig, path)
    plt.show()



In [ ]:
set_prices = set_price_comparison(pokemon_all, stat="max")


title = "Ungraded: Max"
subtitle = "Standard: Any"
path = "images/beeswarm_test.png"
source = "Source: Test source."
note = "Note: Test note."

decade_beeswarm(set_prices, title, subtitle, path, highlight=(), note=note, source=source)

In [ ]:
set_prices = set_price_comparison(pokemon_all, stat="max")


title = "Ungraded: Max"
subtitle = "Standard: Any"
path = "images/beeswarm_test.png"
source = "Source: Test source."
note = "Note: Test note."

decade_beeswarm(set_prices, title, subtitle, path, highlight=(), note=note, source=source)